# FoldJAX on Google Colab

Run a complete JAX structure prediction from a protein sequence, inspect the confidence scores, view the predicted structure, and download the result. This quickstart uses the public **Protenix** checkpoint because it is FoldJAX's smallest complete, publicly fetchable managed model bundle (about 2.1 GB).

Before running:

1. Choose **Runtime → Change runtime type → GPU**. The short example is designed for standard NVIDIA Colab runtimes; GPU type, availability, and runtime vary by account and session.
2. Start from a fresh runtime and run the cells in order. The install cell replaces Colab's preinstalled JAX with FoldJAX's pinned CUDA 12 build from an immutable Git revision, then restarts the runtime once so no preinstalled numerical modules remain mixed. Reconnect and run the notebook from the first cell again.
3. The first prediction includes checkpoint conversion and JAX compilation, so it is much slower than a repeated run of the same shape. Colab runtimes and local disks are ephemeral unless you enable the optional Drive cache below.

> **Scientific scope:** `FAST_DEMO=True` uses 1 sample, 20 diffusion steps, and 1 recycle to keep the tutorial short. It is not the released Protenix schedule. Set it to `False` for the model's released 5-sample, 200-step, 10-recycle defaults. The notebook also defaults to `MSA_POLICY="none"`, so it does not send the sequence to an external MSA service, but prediction quality may be lower than with an alignment. Colab itself is a hosted service, and saved/shared notebooks or result archives can retain input information.


In [ ]:
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path

FOLDJAX_REF = "49800135a81f6d6e9cacaee547d1ea09c083f94b"
install_marker = Path(f"/content/.foldjax-colab-{FOLDJAX_REF}.installed")

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"FoldJAX requires Python 3.12; this runtime has {sys.version.split()[0]}."
    )
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime was found. Select Runtime → Change runtime "
        "type → GPU and reconnect before installing."
    )

if not install_marker.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            (
                "foldjax[cuda12] @ git+https://github.com/eightmm/"
                f"FoldJAX.git@{FOLDJAX_REF}"
            ),
            "py3Dmol>=2.0,<3",
        ]
    )
    install_marker.write_text(FOLDJAX_REF)
    print("Dependencies installed; restarting the runtime once...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print(f"FoldJAX dependencies are installed from {FOLDJAX_REF[:12]}.")

## Runtime and cache location

Leave `PERSIST_TO_DRIVE=False` for the fastest one-off run. Set it to `True` before running this cell if you want the downloaded and converted weights to survive a Colab runtime reset. Google Drive is slower than the local Colab disk, especially during checkpoint loading. The device-specific JAX compilation cache stays on the local runtime disk.


In [ ]:
PERSIST_TO_DRIVE = False
WORK_DIR = Path("/content/foldjax-colab")

if PERSIST_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    foldjax_home = Path("/content/drive/MyDrive/foldjax-cache")
else:
    foldjax_home = Path("/content/foldjax-cache")

os.environ["FOLDJAX_HOME"] = str(foldjax_home)
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")
os.environ["PROTENIX_TRIANGLE_MULTIPLICATION_BACKEND"] = "xla"
WORK_DIR.mkdir(parents=True, exist_ok=True)
foldjax_home.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(foldjax_home).free
if free_bytes < 8_000_000_000:
    raise RuntimeError(
        f"Only {free_bytes / 1e9:.1f} GB is free; keep at least 8 GB for "
        "the checkpoint, its converted copy, and working files."
    )
print(f"FoldJAX home: {foldjax_home} ({free_bytes / 1e9:.1f} GB free)")

In [ ]:
import jax

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
print(f"JAX {jax.__version__}: {jax.devices()}")
if jax.__version__ != "0.10.1":
    raise RuntimeError(
        f"Expected JAX 0.10.1 but imported {jax.__version__}. Restart the "
        "runtime and run the install cell again."
    )
gpu_info = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=False,
    capture_output=True,
    text=True,
)
if gpu_info.stdout.strip():
    print(gpu_info.stdout.strip())
if not gpu_devices:
    raise RuntimeError(
        "No JAX GPU was detected. Select Runtime → Change runtime type → GPU, "
        "restart the runtime, and run the notebook from the first cell."
    )

## Download and prepare public weights

This downloads the publisher's pinned Protenix checkpoint and supporting assets, verifies their SHA-256 digests, and converts the checkpoint to FoldJAX's native NumPy/JAX archive. Re-running the cell skips already verified files. No model weights are stored in this notebook or redistributed by FoldJAX. The next cell prints the publisher, licence, and registered download size before starting.


In [ ]:
from foldjax import model_info

info = model_info("protenix")
print(f"Source: {info.weights_source}")
print(f"Licence: {info.weights_licence}")
print(f"Download: {info.download_bytes / 1e9:.2f} GB")
subprocess.run(
    [
        sys.executable,
        "-m",
        "foldjax.cli",
        "weights",
        "fetch",
        "--model",
        "protenix",
    ],
    check=True,
)

## Build a protein job

The example is human ubiquitin (76 residues). `MSA_POLICY="none"` performs single-sequence prediction without sending the sequence to an additional MSA service. If you deliberately change it to `"auto"`, FoldJAX sends the sequence to the public ColabFold MMseqs2 server and caches the returned alignment. Colab remains a hosted service either way; use sensitive sequences only if its storage and sharing policy is acceptable to you.


In [ ]:
from foldjax import Job

SEQUENCE = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTL"
    "SDYNIQKESTLHLVLRLRGG"
)
FAST_DEMO = True
MSA_POLICY = "none"

job = Job.from_sequences(protein=(SEQUENCE,), name="ubiquitin")
job_path = job.write(WORK_DIR / "ubiquitin.json")
print(f"Wrote {job_path} ({len(SEQUENCE)} residues).")

## Predict

The tutorial forces portable float32 and XLA attention/triangle kernels instead of hardware-specific fused paths. This is slower than the released BF16/fused defaults on newer GPUs but avoids assuming that the Colab runtime supports them. `resume=True` restores an already completed result instead of repeating it when you rerun the cell.


In [ ]:
import json

from foldjax import PredictionRequest, predict, progress

progress.enable()
demo_schedule = (
    {"num_samples": 1, "num_steps": 20, "num_recycles": 1}
    if FAST_DEMO
    else {}
)
schedule_label = "fast-demo" if FAST_DEMO else "released"
request = PredictionRequest(
    model="protenix",
    input=job_path,
    output_dir=WORK_DIR / "outputs" / f"ubiquitin-protenix-{schedule_label}",
    cache_dir=WORK_DIR / "compile-cache",
    seed=0,
    msa=MSA_POLICY,
    options={
        "dtype": "float32",
        "attention_kernel": "xla",
        "triangle_kernel": "xla",
    },
    resume=True,
    **demo_schedule,
)
result = predict(request)
print(json.dumps(result.summary(), indent=2))

In [ ]:
import math

import gemmi

sample = result.samples[0]
if sample.structure_path is None:
    raise RuntimeError("Prediction completed without a structure file.")
structure_path = Path(sample.structure_path)
if not structure_path.is_file() or structure_path.stat().st_size == 0:
    raise RuntimeError(f"Missing or empty structure: {structure_path}")
if not sample.scores or not all(
    math.isfinite(float(value)) for value in sample.scores.values()
):
    raise RuntimeError(
        "Prediction returned invalid confidence scores: "
        f"{sample.scores}"
    )
if structure_path.suffix.lower() in {".cif", ".mmcif"}:
    gemmi.cif.read_file(str(structure_path))
else:
    gemmi.read_structure(str(structure_path))
print(f"Structure: {structure_path}")
for name, value in sorted(sample.scores.items()):
    print(f"{name}: {value:.4f}")

## View the structure


In [ ]:
import py3Dmol

structure_format = (
    "cif" if structure_path.suffix.lower() in {".cif", ".mmcif"} else "pdb"
)
viewer = py3Dmol.view(width=900, height=560)
viewer.addModel(structure_path.read_text(), structure_format)
viewer.setStyle({"cartoon": {"color": "spectrum"}})
viewer.setBackgroundColor("white")
viewer.zoomTo()
viewer.show()

## Download all outputs

The archive contains the predicted structure, confidence files, and `foldjax_run.json`, which records the exact input, weights, options, timings, and artifacts for the run.


In [ ]:
import shutil

if result.output_dir is None:
    raise RuntimeError("Prediction completed without an output directory.")
archive_path = Path(
    shutil.make_archive(
        str(WORK_DIR / "foldjax-ubiquitin-protenix"),
        "zip",
        root_dir=result.output_dir,
    )
)
print(f"Created {archive_path}")
try:
    from google.colab import files

    files.download(str(archive_path))
except ImportError:
    print("Download this archive from your notebook file browser.")

## Next steps

- Set `FAST_DEMO=False` and rerun from **Build a protein job** for the released Protenix sampling schedule. This costs substantially more compute.
- Replace `SEQUENCE` with your own protein. Keep `MSA_POLICY="none"` to avoid sending it to an additional MSA service, or explicitly opt into `"auto"` after accepting that transfer. Colab and any saved/shared notebook or result archive still handle the sequence.
- Enable `PERSIST_TO_DRIVE` before the runtime cell if you want the weights to survive session resets. The compilation cache remains local because entries are tied to the JAX version, device, weights, options, and concrete input shape.
- See the [FoldJAX README](https://github.com/eightmm/FoldJAX) for Boltz-2, ESMFold2, OpenDDE, OpenFold3, and AlphaFold 3 setup, ligands, nucleic acids, templates, batch prediction, and full sampling guidance.
